# Lecture: DDPG and TD3 from stable-baselines3

**Objective**: Apply the Deep Deterministic Policy Gradient (DDPG)
implementation of `stable-baselines3` (SB3) to the `Pendulum-v1` environment
&mdash; already familiar from the continuous-action actor-critic in ch7, but
this time trained **off-policy** &mdash; and learn to **read the off-policy
training metrics**:

- how they differ from the on-policy metrics of PPO (ch8),
- what a healthy DDPG run looks like,
- how to recognise the **overestimation bias** that makes DDPG brittle,
- and how **TD3**'s three fixes (clipped double-Q, delayed policy updates,
  target policy smoothing) address exactly that instability.

`Pendulum-v1` has a 3-dimensional observation and a 1-dimensional continuous
action (torque). The reward is always negative, with **0** as the
(unreachable) optimum, so "solved vs. not solved" is easy to read off a plot.

Run the following cell **only on Google Colab** to install
`stable-baselines3`. If you work locally and already installed it (manually or
via `requirements.txt`), skip this cell.

In [ ]:
!pip install stable-baselines3==2.6.0

### Exercise 1: Train DDPG, log the metrics and plot them

The cell below trains SB3's DDPG on `Pendulum-v1` with the default
**`MlpPolicy`**. All hyperparameters that matter for training time and
stability are collected in one `DDPG_CONFIG` dict so you can see and tweak
them in a single place. With `verbose=1`, DDPG prints a table every few
episodes; the `MetricLogger` callback additionally records the key metrics so
we can plot them afterwards with `matplotlib` &mdash; this works identically
**locally and on Colab**, no external tools needed.

The default config below converges on Pendulum in roughly 5 minutes on a
Colab CPU/GPU runtime.

**Task**: While training runs, compare the printed table to the PPO table
from ch8. Which metrics are gone? Which are new? Why (think replay buffer,
deterministic actor, no clipping)?

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import DDPG
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy


class MetricLogger(BaseCallback):
    """Collect selected metrics from SB3's logger during training."""

    def __init__(self, log_every=200):
        super().__init__()
        self.log_every = log_every
        self.records = {"timesteps": [], "ep_rew_mean": [], "actor_loss": [],
                        "critic_loss": []}

    def _on_step(self) -> bool:
        if self.num_timesteps % self.log_every == 0:
            logs = self.logger.name_to_value
            ep_buffer = self.model.ep_info_buffer
            ep_rew = np.mean([e["r"] for e in ep_buffer]) if len(ep_buffer) > 0 else np.nan
            self.records["timesteps"].append(self.num_timesteps)
            self.records["ep_rew_mean"].append(ep_rew)
            self.records["actor_loss"].append(logs.get("train/actor_loss", np.nan))
            self.records["critic_loss"].append(logs.get("train/critic_loss", np.nan))
        return True


def plot_metrics(records, title_suffix="", color="tab:blue"):
    panels = [
        ("ep_rew_mean", "Mean episodic reward", 0, "target -> 0"),
        ("actor_loss", "actor_loss", None, "should decrease"),
        ("critic_loss", "critic_loss", None, "should trend down, stay bounded"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, (key, title, ref, note) in zip(axes, panels):
        ax.plot(records["timesteps"], records[key], color=color)
        if ref is not None:
            ax.axhline(ref, color="grey", ls="--", alpha=0.7, label=note)
            ax.legend(fontsize=8)
        ax.set_title(title + title_suffix)
        ax.set_xlabel("timesteps")
        ax.grid(True, ls="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
# All hyperparameters that affect training time and stability, in one place --
# see the "DDPG_CONFIG" reference table below the training cell for details on
# each entry's effect.
TOTAL_TIMESTEPS = 25_000

DDPG_CONFIG = dict(
    # Step size for both actor and critic optimizers. Too high pushes the
    # critic into overestimating faster than it can self-correct (Exercise 2).
    learning_rate=1e-3,

    # Replay buffer capacity in transitions. Only affects memory usage here,
    # 1M is far more than Pendulum's ~25k-step budget will ever fill.
    buffer_size=1_000_000,

    # Steps of random actions collected before training starts, so the
    # replay buffer has some diversity before the critic first regresses.
    learning_starts=100,

    # Minibatch size sampled from the replay buffer per gradient update.
    batch_size=256,

    # Polyak averaging coefficient for target networks: target <- tau * online
    # + (1 - tau) * target. Larger tau tracks the online network faster but
    # is less stable (the DQN-style target-freeze effect weakens).
    tau=0.005,

    # Discount factor.
    gamma=0.99,

    # How much data to collect between train() calls -- one full episode here.
    train_freq=(1, "episode"),

    # Number of SGD updates per train() call. This is the main lever for
    # wall-clock time: DDPG's default (-1) means one update per collected
    # step, i.e. ~200 updates per Pendulum episode. Capping it at 64 is most
    # of the speedup relative to the SB3 default.
    gradient_steps=64,

    # Exploration noise added to actions during rollout (DDPG's actor is
    # deterministic and has no exploration of its own, unlike SAC). Set to a
    # NormalActionNoise(...) instance to override the SB3 default.
    action_noise=None,
)

In [ ]:
# Create the environment
env = gym.make("Pendulum-v1")

# Create a DDPG agent with an MLP policy. tensorboard_log is optional (see below).
logger_cb = MetricLogger()
model = DDPG("MlpPolicy", env, verbose=1, tensorboard_log="./ddpg_pendulum_tb/", **DDPG_CONFIG)

# Train the model
model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=logger_cb)

# Evaluate the trained policy over 10 episodes
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=10)
print(f"\nMean Reward: {mean_reward:.2f} +/- {std_reward:.2f}")

# Save the model
model.save("ddpg_pendulum")

plot_metrics(logger_cb.records)

### Reference: what the DDPG training metrics mean

On **Pendulum-v1** the reward per step lies in $[-16.27, 0]$, so episodic
return over 200 steps is at best close to **0** and typically **-1500 to
-200** for a well-trained agent (the pendulum cannot be held perfectly upright
at all times because of the reward shaping, so small negative returns are
normal even for a good policy).

#### `rollout/` &mdash; behaviour in the environment

| Metric | Meaning | Normal range on Pendulum | What to watch for |
|--------|---------|---------------------------|--------------------|
| `ep_rew_mean` | Mean episodic return over recent episodes | starts ~-1200 to -1500, should climb toward **-200 or better** | Flat/decreasing &rarr; learning stalled or diverged |
| `ep_len_mean` | Mean episode length | constant **200** (fixed horizon, no early termination) | &mdash; |

#### `time/` &mdash; bookkeeping

| Metric | Meaning | Notes |
|--------|---------|-------|
| `fps` | Environment steps per second | Hardware-dependent; only relative changes matter |
| `episodes` | Number of completed episodes so far | &mdash; |
| `total_timesteps` | Environment steps collected so far | Progress toward your budget |

#### `train/` &mdash; the optimisation itself (the important diagnostics)

| Metric | Meaning | Normal / healthy range | What it tells you |
|--------|---------|--------------------------|--------------------|
| `actor_loss` | $-\,\mathbb E_s[Q_\phi(s,\mu_\theta(s))]$, the actor climbs the critic | should **decrease** (become more negative) as the actor finds better actions | Flat/increasing &rarr; actor not improving, or critic collapsed |
| `critic_loss` | TD-error MSE of the critic on the Bellman target | should **trend down**, some noise is normal | Exploding &rarr; overestimation bias / divergence (see below) |
| `learning_rate` | Current optimizer step size | set via `DDPG_CONFIG["learning_rate"]`, default **1e-3** | Changes only if you schedule it |
| `n_updates` | Total gradient updates so far | grows by `gradient_steps` per `train_freq` interval | &mdash; |

**What is missing compared to PPO (ch8), and why:**
- no `approx_kl`, `clip_fraction`, `clip_range` &mdash; there is no trust
  region and no importance-sampling ratio; DDPG is natively off-policy
  (ch9 slides, "no importance weights, no ratios")
- no `entropy_loss` &mdash; the actor $\mu_\theta$ is **deterministic**, there
  is no distribution to measure entropy of; exploration instead comes from
  external action noise added during data collection

The comments in the `DDPG_CONFIG` cell above explain what each entry does and
why; `gradient_steps` is the one to remember if training feels slow, and
`learning_rate` / `action_noise` are the ones Exercise 2 destabilises.

**Rules of thumb for a healthy DDPG run:**
- `ep_rew_mean` rises steadily and then plateaus at a good (near-zero,
  strongly negative-but-bounded) value.
- `critic_loss` trends down and stays bounded &mdash; a late explosion is the
  signature of overestimation bias feeding back into the TD target.
- `actor_loss` keeps decreasing as long as the critic estimate is trustworthy.

#### Optional: TensorBoard

Because the training cell passes `tensorboard_log="./ddpg_pendulum_tb/"`, you
can also inspect the curves in TensorBoard.

- **Locally**: run `tensorboard --logdir ./ddpg_pendulum_tb/` in a terminal and
  open the printed URL.
- **On Colab**: run the two lines below in a cell.

```python
%load_ext tensorboard
%tensorboard --logdir ./ddpg_pendulum_tb/
```

The inline matplotlib plots above already cover the essentials, so TensorBoard
is optional here.

### Exercise 2: Making DDPG brittle on purpose

The slides claim DDPG is **brittle** and prone to **overestimation bias**: the
critic has approximation errors, and the actor is trained to seek out exactly
the actions where the critic errs upward.

The run above is usually well-behaved. To actually *see* the failure mode, we
override two entries of `DDPG_CONFIG`: a much higher `learning_rate` and
weaker `action_noise`. An over-confident, under-explored critic overestimates
faster than it can correct itself.

**Task**: run the cell below (same timestep budget as Exercise 1), then
compare its `critic_loss` and `ep_rew_mean` curves to the healthy run above.
Where does `critic_loss` start growing instead of shrinking? Does
`ep_rew_mean` collapse at the same point?

In [ ]:
from stable_baselines3.common.noise import NormalActionNoise

env = gym.make("Pendulum-v1")
n_actions = env.action_space.shape[-1]

# Small action noise -> little exploration; large critic learning rate ->
# critic overfits fast to its own (biased) targets.
unstable_config = DDPG_CONFIG | dict(
    learning_rate=1e-2,
    action_noise=NormalActionNoise(mean=np.zeros(n_actions), sigma=0.02 * np.ones(n_actions)),
)

unstable_logger_cb = MetricLogger()
unstable_model = DDPG("MlpPolicy", env, verbose=0, **unstable_config)
unstable_model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=unstable_logger_cb)

plot_metrics(unstable_logger_cb.records, title_suffix=" (destabilised)", color="tab:red")

### Exercise 3: TD3 &mdash; fixing DDPG's instability

TD3 (Fujimoto et al. 2018) is DDPG plus three fixes against overestimation:

1. **Clipped double-Q**: train *two* critics $Q_{\phi_1}, Q_{\phi_2}$ and use
   the **minimum** in the TD target,
   $y = r + \gamma \min_{i=1,2} Q_{\phi_i^{-}}(s', a')$ &mdash; pessimism
   instead of optimism.
2. **Delayed policy updates**: update the actor (and targets) only every $d$
   critic updates &mdash; let the critic settle before the actor exploits it.
3. **Target policy smoothing**: add clipped noise to the target action,
   $a' = \mu_{\theta^{-}}(s') + \text{clip}(\varepsilon, -c, c)$ &mdash; the
   critic cannot form sharp spurious peaks.

SB3's `TD3` shares almost the entire DDPG API, so `TD3_CONFIG` below is just
`DDPG_CONFIG` plus the three new parameters (`policy_delay`,
`target_policy_noise`, `target_noise_clip`) implementing fixes 2 and 3; fix 1
(the second critic) is built into `TD3`'s network architecture and needs no
extra parameter.

**Task**: run the cell below with the *same* destabilising overrides as
Exercise 2 (high `learning_rate`, weak `action_noise`) &mdash; conditions
under which DDPG diverged. Does TD3's `critic_loss` stay bounded this time?
Compare the three plots directly to the red (destabilised DDPG) curves from
Exercise 2.

In [ ]:
from stable_baselines3 import TD3

env = gym.make("Pendulum-v1")
n_actions = env.action_space.shape[-1]

# Same destabilising overrides as Exercise 2 -- TD3 should stay stable anyway.
TD3_CONFIG = DDPG_CONFIG | dict(
    learning_rate=1e-2,
    action_noise=NormalActionNoise(mean=np.zeros(n_actions), sigma=0.02 * np.ones(n_actions)),
    policy_delay=2,           # fix 2: update actor + targets every d critic updates
    target_policy_noise=0.2,  # fix 3: stddev of the clipped noise added to the target action
    target_noise_clip=0.5,    # fix 3: clip range for that noise
)

td3_logger_cb = MetricLogger()
td3_model = TD3("MlpPolicy", env, verbose=0, **TD3_CONFIG)
td3_model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=td3_logger_cb)

plot_metrics(td3_logger_cb.records, title_suffix=" (TD3, destabilising config)", color="tab:green")

### Exercise 4: Watch a trained episode

Test the (well-trained, `model` from Exercise 1) agent on one Pendulum
episode.

#### Run this if you use a local Python setup:

In [ ]:
import gymnasium as gym
from stable_baselines3 import DDPG

env = gym.make("Pendulum-v1", render_mode="human")

# Load the model and run one episode
model = DDPG.load("ddpg_pendulum")

obs, _ = env.reset()
done = False
while not done:
    action, _ = model.predict(obs)
    obs, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated
    env.render()

env.close()

#### Run this if you use Colab (renders to a video):

In [ ]:
import gymnasium as gym
from stable_baselines3 import DDPG

from IPython.display import HTML
from base64 import b64encode

import imageio

# instantiation of the environment
env = gym.make("Pendulum-v1", render_mode="rgb_array")

# resetting the environment for first start
obs, _ = env.reset()

# initialize a list of frames for video creation
frames = []

done = False
while not done:
    # capture the frame and append it to frames list
    frame = env.render()
    frames.append(frame)

    action, _ = model.predict(obs)
    # do one step in the environment
    obs, reward, terminated, truncated, info = env.step(action)

    # flag whether the episode is finished
    done = terminated or truncated

    # final rendering for last image of episode
    if done:
      frame = env.render()
      frames.append(frame)

env.close()

# save video as
video_path = "./Pendulum_vid_own_policy.mp4"
imageio.mimsave(video_path, frames, fps=30)

In [ ]:
# this is for displaying the video after saving
mp4 = open(video_path, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width=400 controls>
    <source src="{data_url}" type="video/mp4">
</video>
""")